# 📚 Error Handling & File I/O (에러 처리 & 파일 I/O)
> try/except/else/finally, built-in and custom exceptions, assert, and reading/writing text and CSV files safely.  
> try/except/else/finally, 내장·커스텀 예외, assert, 그리고 텍스트와 CSV 파일을 안전하게 읽고 쓰기.

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn: (오늘 배우고 싶은 것)
- [x] Write `try/except/else/finally` blocks that catch specific exception types, and know when to use `else` vs `finally`.  
특정 예외 타입을 잡는 `try/except/else/finally` 블록을 작성하고, `else`와 `finally`를 언제 써야 하는지 안다.
- [x] Recognize Python's most common built-in exceptions on sight, and write a custom exception class when a built-in one is not specific enough.  
Python의 가장 흔한 내장 예외를 보자마자 알아보고, 내장 예외가 충분히 구체적이지 않을 때 커스텀 예외 클래스를 작성한다.
- [x] Read and write text and CSV files safely using `with open(...)`, and know the difference between the file modes (`r`, `w`, `a`).  
`with open(...)`을 사용해 텍스트와 CSV 파일을 안전하게 읽고 쓰며, 파일 모드(`r`, `w`, `a`)의 차이를 안다.

---
# 🧠 Concept (개념)

## What is it? (이게 뭔가요?)
*(Explain it in your own words. 자신의 언어로 설명해보세요.)*

Error handling in Python centers on `try/except` blocks that catch specific exception classes — `ValueError`, `KeyError`, `FileNotFoundError`, and dozens more — rather than a single generic error type. File I/O centers on the `with open(...) as f:` pattern, which guarantees the file gets closed automatically even if an error happens partway through reading or writing it.

Python의 에러 처리는 하나의 범용 에러 타입이 아니라 `ValueError`, `KeyError`, `FileNotFoundError`를 비롯한 수십 개의 구체적인 예외 클래스를 잡는 `try/except` 블록을 중심으로 돌아간다. 파일 I/O는 `with open(...) as f:` 패턴을 중심으로 돌아가며, 이는 읽거나 쓰는 도중 에러가 발생해도 파일이 자동으로 닫히는 것을 보장한다.

| | JS/TS | Python |
|---|---|---|
| Catch an error 에러 잡기 | `try { } catch (e) { }` | `try: ... except Exception as e: ...` |
| Catch a SPECIFIC error type 특정 에러 타입 잡기 | `if (e instanceof TypeError)` inside one catch | `except TypeError as e:` — a separate clause |
| Always run cleanup code 항상 정리 코드 실행 | `finally { }` | `finally:` (same idea!) |
| Throw an error 에러 던지기 | `throw new Error("msg")` | `raise ValueError("msg")` |
| Custom error class 커스텀 에러 클래스 | `class MyError extends Error` | `class MyError(Exception):` |
| Read a file 파일 읽기 | `fs.readFileSync(path, "utf8")` | `open(path).read()` (ideally inside `with`) |
| Auto-close a resource 리소스 자동 종료 | manual `try/finally` or none at all | `with open(...) as f:` |

The biggest structural difference: JS/TS usually has ONE `catch` block that checks the error's type manually with `instanceof`, while Python has multiple `except` clauses, each targeting a specific exception class directly — closer to a `switch` on error type than a single catch-all.

가장 큰 구조적 차이: JS/TS는 보통 `instanceof`로 에러 타입을 수동으로 확인하는 `catch` 블록 하나를 갖지만, Python은 각각 특정 예외 클래스를 직접 대상으로 하는 여러 `except` 절을 가진다 — 하나의 catch-all보다는 에러 타입에 대한 `switch`에 가깝다.

## Why do we use it? (왜 사용하나요?)
*(When is it useful? 언제 유용한가요?)*

Catching a specific exception type (`except ValueError:`) instead of everything (`except Exception:`) means a genuinely unexpected bug still crashes loudly and gets noticed, while only the errors you actually anticipated get silently handled — catching too broadly hides real bugs behind a generic error message.

모든 것(`except Exception:`) 대신 특정 예외 타입(`except ValueError:`)을 잡는 것은, 진짜로 예상치 못한 버그는 여전히 크게 터져서 알아차릴 수 있고, 실제로 예상했던 에러만 조용히 처리된다는 뜻이다 — 너무 넓게 잡으면 실제 버그가 범용 에러 메시지 뒤에 숨는다.

The `with` statement for files exists because forgetting to close a file (or skipping it due to an early exception) leaks a file handle and, in write mode, can leave data only partially written to disk — `with` guarantees the file closes no matter how the block exits, error or not.

파일에 대한 `with` 문이 존재하는 이유는 파일을 닫는 것을 잊거나(또는 이른 예외로 인해 건너뛰어서) 파일 핸들이 누수되고, 쓰기 모드에서는 데이터가 디스크에 부분적으로만 기록될 수 있기 때문이다 — `with`는 블록이 에러로 끝나든 정상적으로 끝나든 파일이 닫히는 것을 보장한다.

## When is it used in Business Analytics? (Business Analytics에서는 언제 쓰이나요?)
*(Real-world use case 실제 사용 사례)*

Real data is messy — a "numeric" column might contain a stray text value, a file a script expects might not exist yet, an API response might be missing a field. Wrapping the risky step in `try/except` and logging or skipping the bad row, instead of letting the whole script crash on row 4,000 of 10,000, is standard practice in any real cleaning pipeline.

실제 데이터는 지저분하다 — "숫자" 컬럼에 엉뚱한 텍스트 값이 들어있을 수도 있고, 스크립트가 기대하는 파일이 아직 없을 수도 있고, API 응답에 필드가 빠져 있을 수도 있다. 위험한 단계를 `try/except`로 감싸고 잘못된 행을 로그로 남기거나 건너뛰는 것은, 10,000개 중 4,000번째 행에서 스크립트 전체가 죽게 두는 대신, 실제 정리 파이프라인에서 표준적인 관행이다.

CSV remains the universal export/import format for spreadsheet tools, so reading it with `csv.DictReader` (before pandas ever gets involved) is one of the most common first steps of any BA script — and remembering that every value comes back as a `str` avoids a whole category of "why is my sum a string" bugs.

CSV는 스프레드시트 도구의 보편적인 내보내기·가져오기 포맷으로 남아 있어서, (pandas가 관여하기 전에) `csv.DictReader`로 읽는 것은 거의 모든 BA 스크립트의 가장 흔한 첫 단계 중 하나다 — 모든 값이 `str`로 돌아온다는 것을 기억하면 "왜 내 합계가 문자열이지" 같은 카테고리의 버그 전체를 피할 수 있다.

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [1]:
# Basic try/except
# 기본 try/except
try:
    value = int("not a number")
except ValueError as e:
    print(f"caught: {e}")

# finally always runs — cleanup code that must happen no matter what
# finally는 항상 실행됨 — 무슨 일이 있어도 실행되어야 하는 정리 코드
try:
    print("doing risky work")
finally:
    print("cleanup happens here, error or not")

# Reading and writing a file with `with` — closes automatically
# with로 파일 읽고 쓰기 — 자동으로 닫힘
with open("quick_demo.txt", "w", encoding="utf-8") as f:
    f.write("hello file\n")

with open("quick_demo.txt", "r", encoding="utf-8") as f:
    print(f.read())

import os
os.remove("quick_demo.txt")           # clean up the demo file (데모 파일 정리)

caught: invalid literal for int() with base 10: 'not a number'
doing risky work
cleanup happens here, error or not
hello file



## Common Variations (자주 쓰는 변형)

In [ ]:
import os

# Catching multiple specific exception types with one clause
# 하나의 절로 여러 특정 예외 타입 잡기
def to_number(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return None

print(to_number("42"), to_number("abc"), to_number(None))

# try/except/else — else runs ONLY when no exception occurred
# try/except/else — else는 예외가 발생하지 않았을 때만 실행됨
for value in ["10", "abc"]:
    try:
        n = int(value)
    except ValueError:
        print(f"{value!r} is not a number")
    else:
        print(f"{value!r} parsed successfully as {n}")

# assert for a quick internal sanity check
# 빠른 내부 확인을 위한 assert
def average(numbers):
    assert len(numbers) > 0, "cannot average an empty list"
    return sum(numbers) / len(numbers)

print(average([1, 2, 3]))

# Reading a file line by line — memory-efficient, works like a generator
# 파일을 한 줄씩 읽기 — 메모리 효율적, 제너레이터처럼 동작
with open("quick_demo2.txt", "w", encoding="utf-8") as f:
    f.writelines(["a\n", "b\n", "c\n"])

with open("quick_demo2.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

os.remove("quick_demo2.txt")

42 None None
'10' parsed successfully as 10
'abc' is not a number
2.0
a
b
c


---
# 🧪 Small Examples (예제)

## Example 1 — try/except/else/finally (기본 예외 처리)

In [3]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError as e:
        print(f"Error caught: {e}")
        return None
    else:
        print("No error — this only runs when the try block succeeds")
        return result
    finally:
        print("This always runs, error or not")

print(safe_divide(10, 2))
print("---")
print(safe_divide(10, 0))

# Catching multiple exception types in one except clause
# 하나의 except 절에서 여러 예외 타입 잡기
def parse_number(value):
    try:
        return int(value)
    except (ValueError, TypeError) as e:
        print(f"Could not parse {value!r}: {e}")
        return None

print(parse_number("42"))
print(parse_number("abc"))
print(parse_number(None))

# Catching Exception broadly — use sparingly, it hides the specific error type
# Exception을 넓게 잡기 — 아껴 써야 함, 구체적인 에러 타입을 감춤
def risky_operation(items, index):
    try:
        return items[index]
    except Exception as e:
        print(f"{type(e).__name__}: {e}")
        return None

print(risky_operation([1, 2, 3], 10))

No error — this only runs when the try block succeeds
This always runs, error or not
5.0
---
Error caught: division by zero
This always runs, error or not
None
42
Could not parse 'abc': invalid literal for int() with base 10: 'abc'
None
Could not parse None: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
None
IndexError: list index out of range
None


## Example 2 — Built-in Exceptions & raise (내장 예외·raise)

In [4]:
# A quick tour of the exceptions you will see most often
# 가장 자주 보게 될 예외들 빠르게 훑어보기
exception_demos = [
    ("ValueError", lambda: int("abc")),
    ("TypeError", lambda: 1 + "a"),
    ("KeyError", lambda: {"a": 1}["z"]),
    ("IndexError", lambda: [1, 2, 3][10]),
    ("AttributeError", lambda: "hello".nonexistent_method()),
    ("ZeroDivisionError", lambda: 1 / 0),
]

for name, trigger in exception_demos:
    try:
        trigger()
    except Exception as e:
        print(f"{name}: {type(e).__name__} - {e}")

# raise — trigger an exception on purpose
# raise — 의도적으로 예외 발생시키기
def set_age(age):
    if age < 0:
        raise ValueError(f"age cannot be negative, got {age}")
    return age

try:
    set_age(-5)
except ValueError as e:
    print("caught:", e)

# Bare raise — re-raise the CURRENT exception, useful inside an except block
# 단독 raise — 현재 예외를 재발생, except 블록 안에서 유용
def process_data(value):
    try:
        return 100 / value
    except ZeroDivisionError:
        print("logging the error before re-raising...")
        raise                # re-raises the same ZeroDivisionError

try:
    process_data(0)
except ZeroDivisionError as e:
    print("final handler caught:", e)

ValueError: ValueError - invalid literal for int() with base 10: 'abc'
TypeError: TypeError - unsupported operand type(s) for +: 'int' and 'str'
KeyError: KeyError - 'z'
IndexError: IndexError - list index out of range
AttributeError: AttributeError - 'str' object has no attribute 'nonexistent_method'
ZeroDivisionError: ZeroDivisionError - division by zero
caught: age cannot be negative, got -5
logging the error before re-raising...
final handler caught: division by zero


## Example 3 — Custom Exceptions & assert (커스텀 예외·assert)

In [5]:
# A custom exception — inherits from Exception (or a more specific built-in one)
# 커스텀 예외 — Exception(또는 더 구체적인 내장 예외)을 상속
class InsufficientFundsError(Exception):
    def __init__(self, balance, amount):
        message = f"balance ${balance} cannot cover withdrawal of ${amount}"
        super().__init__(message)
        self.balance = balance
        self.amount = amount

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

try:
    withdraw(50, 100)
except InsufficientFundsError as e:
    print("caught:", e)
    print("shortfall:", e.amount - e.balance)          # custom exceptions can carry extra data (커스텀 예외는 추가 데이터를 담을 수 있음)

# A custom exception hierarchy — catch broadly OR specifically
# 커스텀 예외 계층 — 넓게 또는 구체적으로 잡기
class ValidationError(Exception):
    pass

class MissingFieldError(ValidationError):
    pass

class InvalidFormatError(ValidationError):
    pass

def validate(record):
    if "email" not in record:
        raise MissingFieldError("email field is required")
    if "@" not in record["email"]:
        raise InvalidFormatError(f"invalid email: {record['email']}")

for record in [{}, {"email": "not-an-email"}, {"email": "ok@example.com"}]:
    try:
        validate(record)
        print("valid:", record)
    except ValidationError as e:                  # catches BOTH subclasses (두 서브클래스 모두 잡음)
        print(f"{type(e).__name__}: {e}")

# assert — a quick sanity check, meant for catching bugs during development
# assert — 개발 중 버그를 잡기 위한 빠른 확인
def calculate_discount(price, percent):
    assert 0 <= percent <= 100, f"percent must be 0-100, got {percent}"
    return price * (1 - percent / 100)

print(calculate_discount(100, 20))
try:
    calculate_discount(100, 150)
except AssertionError as e:
    print("caught:", e)

caught: balance $50 cannot cover withdrawal of $100
shortfall: 50
MissingFieldError: email field is required
InvalidFormatError: invalid email: not-an-email
valid: {'email': 'ok@example.com'}
80.0
caught: percent must be 0-100, got 150


## Example 4 — File Reading & Writing (파일 읽기·쓰기)

In [6]:
import os

demo_path = "demo_notes.txt"

# Writing — "w" overwrites the whole file; the `with` block auto-closes it
# 쓰기 — "w"는 파일 전체를 덮어씀; with 블록이 자동으로 닫아줌
with open(demo_path, "w", encoding="utf-8") as f:
    f.write("Line one\n")
    f.writelines(["Line two\n", "Line three\n"])

# Reading the whole file as one string
# 파일 전체를 하나의 문자열로 읽기
with open(demo_path, "r", encoding="utf-8") as f:
    content = f.read()
print(repr(content))

# Reading line by line — memory-efficient for large files (Section 8's lazy idea, applied to files)
# 한 줄씩 읽기 — 큰 파일에도 메모리 효율적 (Section 8의 게으른 아이디어를 파일에 적용)
with open(demo_path, "r", encoding="utf-8") as f:
    for line in f:
        print("line:", line.strip())

# "a" appends instead of overwriting
# "a"는 덮어쓰지 않고 추가
with open(demo_path, "a", encoding="utf-8") as f:
    f.write("Line four (appended)\n")

with open(demo_path, "r", encoding="utf-8") as f:
    print(f.readlines())

# Clean up the demo file
# 데모 파일 정리
os.remove(demo_path)
print(os.path.exists(demo_path))

'Line one\nLine two\nLine three\n'
line: Line one
line: Line two
line: Line three
['Line one\n', 'Line two\n', 'Line three\n', 'Line four (appended)\n']
False


## Practice — CSV Files (연습 — CSV 파일)
Fill in every `________` blank below, then run the cell. It will not raise a syntax error either way — but it WILL raise an error at the first blank you have not filled in yet, which is expected. Fix one, re-run, then move to the next.

아래의 모든 `________` 빈칸을 채운 뒤 셀을 실행하세요. 어느 쪽이든 문법 에러는 나지 않지만, 아직 채우지 않은 첫 번째 빈칸에서 에러가 발생합니다 — 의도된 동작입니다. 하나씩 고치고 다시 실행하며 다음으로 넘어가세요.

In [9]:
# ============================================================
# Practice — CSV Files (연습 — CSV 파일)
# ============================================================
import csv, os

demo_csv_path = "demo_data.csv"

# 1) Writing a CSV with DictWriter — fieldnames define the column order
#    DictWriter로 CSV 쓰기 — fieldnames가 컬럼 순서를 정함
with open(demo_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age"])               # TODO: which list of column names?
    writer.writeheader()
    writer.writerow({"name": "Alice", "age": 25})
    writer.writerow({"name": "Bob", "age": 30})

# 2) Reading a CSV with DictReader — each row becomes a dict, keyed by header
#    DictReader로 CSV 읽기 — 각 행이 헤더를 키로 하는 딕셔너리가 됨
with open(demo_csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)                                        # TODO: which csv class reads rows as dicts?
    rows = list(reader)
print("2)", rows)                                                     # Expected: [{'name': 'Alice', 'age': '25'}, {'name': 'Bob', 'age': '30'}]

# 3) CSV values are always read back as strings — convert manually when needed
#    CSV 값은 항상 문자열로 읽힘 — 필요하면 직접 변환해야 함
first_age_as_int = int(rows[0]["age"])
print("3)", first_age_as_int, type(first_age_as_int).__name__)          # TODO: which function converts a numeric string to int? Expected: 25 int

# 4) writerows() writes several rows at once, instead of calling writerow() in a loop
#    writerows()는 반복문에서 writerow()를 호출하는 대신 여러 행을 한 번에 씀
extra_people = [{"name": "Kim", "age": 40}, {"name": "Lee", "age": 22}]
with open(demo_csv_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age"])
    writer.writerows(extra_people)                                        # TODO: which method writes MULTIPLE rows at once?

with open(demo_csv_path, "r", encoding="utf-8") as f:
    final_rows = list(csv.DictReader(f))
print("4)", len(final_rows))                                                # Expected: 4  (Alice, Bob, Kim, Lee)

# 5) Clean up the demo file when finished
#    끝나면 데모 파일 정리
os.remove(demo_csv_path)                                                    # TODO: which os function deletes a file?
print("5)", os.path.exists(demo_csv_path))                                      # Expected: False

2) [{'name': 'Alice', 'age': '25'}, {'name': 'Bob', 'age': '30'}]
3) 25 int
4) 4
5) False


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Catching `Exception` too broadly
`except Exception:` catches almost everything, including bugs that have nothing to do with what the `try` block was actually worried about — a typo that raises `NameError`, or a logic error that raises `AttributeError`, gets silently swallowed right alongside the error that was actually expected, making real bugs much harder to notice and debug.

`except Exception:`은 `try` 블록이 실제로 걱정했던 것과 전혀 상관없는 버그까지 거의 모든 것을 잡는다 — 오타로 인한 `NameError`나 로직 에러로 인한 `AttributeError`가, 실제로 예상했던 에러와 나란히 조용히 삼켜져서, 진짜 버그를 알아차리고 디버깅하기 훨씬 어렵게 만든다.

✅ **Fix / 해결법:**  
Catch the most specific exception type you actually expect; reserve broad `except Exception:` for a true top-level safety net, usually with logging.  
실제로 예상하는 가장 구체적인 예외 타입을 잡자; 넓은 `except Exception:`은 보통 로깅과 함께, 진짜 최상위 안전망을 위해서만 남겨두자.
```python
try:
    price = float(row["price"])
except ValueError:          # ✅ specific — only catches the parsing problem
    price = 0.0
```

### Mistake 2 — Not using `with` when working with files
Opening a file with `f = open(path)` and manually calling `f.close()` later is easy to forget, especially if an exception happens between the two lines — the file handle leaks, and in write mode, buffered data might never actually reach the disk.

`f = open(path)`로 파일을 열고 나중에 수동으로 `f.close()`를 호출하는 것은, 특히 두 줄 사이에 예외가 발생하면 잊기 쉽다 — 파일 핸들이 누수되고, 쓰기 모드에서는 버퍼링된 데이터가 실제로 디스크에 도달하지 못할 수도 있다.

✅ **Fix / 해결법:**  
Always use `with open(...) as f:` — it closes the file automatically, even if an exception occurs inside the block.  
항상 `with open(...) as f:`를 사용하자 — 블록 안에서 예외가 발생해도 파일이 자동으로 닫힌다.
```python
with open("file.txt", "r", encoding="utf-8") as f:     # ✅ always closes, even on error
    content = f.read()
```

### Mistake 3 — Forgetting that every CSV value is read back as a `str`
`csv.DictReader` and `csv.reader` never guess types — a column of ages comes back as `"25"`, `"30"`, etc. (strings), not `25`, `30` (ints). Adding two of these values with `+` concatenates them as text instead of summing them.

`csv.DictReader`와 `csv.reader`는 절대 타입을 추측하지 않는다 — 나이 컬럼은 `25`, `30`(정수)이 아니라 `"25"`, `"30"`(문자열)로 돌아온다. 이 값 둘을 `+`로 더하면 합산하는 대신 텍스트로 이어붙인다.

✅ **Fix / 해결법:**  
Convert with `int()` or `float()` immediately after reading a numeric column from CSV.  
CSV에서 숫자 컬럼을 읽은 직후 `int()`나 `float()`로 변환하자.
```python
age = int(row["age"])   # ✅ convert right away, before doing any math
```

---
# 💡 Tips (팁)
Useful tips or shortcuts (유용한 팁이나 단축법)

- Order `except` clauses from most specific to most general — Python checks them top to bottom and stops at the first match, so a broad `except Exception:` placed first would swallow everything below it.  
`except` 절은 구체적인 것부터 일반적인 것 순서로 배치하자 — Python은 위에서 아래로 확인하며 첫 매칭에서 멈추므로, 넓은 `except Exception:`을 먼저 두면 그 아래 모든 것을 삼켜버린다.
- `finally` runs even if the `try` block hits a `return` — it is the right place for cleanup code (closing a connection, releasing a lock) that must happen no matter how the function exits.  
`finally`는 `try` 블록이 `return`을 만나도 실행된다 — 함수가 어떻게 종료되든 반드시 실행되어야 하는 정리 코드(연결 닫기, 잠금 해제)에 적합한 위치다.
- A custom exception is worth creating the moment you find yourself checking error text with `in` to distinguish causes — a dedicated exception class with its own attributes is far more reliable.  
에러 원인을 구분하기 위해 에러 텍스트를 `in`으로 확인하고 있는 자신을 발견하는 순간 커스텀 예외를 만들 가치가 있다 — 자신만의 속성을 가진 전용 예외 클래스가 훨씬 신뢰할 수 있다.
- `assert` statements can be stripped out entirely when Python runs with the `-O` (optimize) flag — never use `assert` for validating untrusted user input; use a real `if ... raise ValueError(...)` for anything that must always be checked.  
`assert` 문은 Python이 `-O`(최적화) 플래그로 실행될 때 완전히 제거될 수 있다 — 신뢰할 수 없는 사용자 입력을 검증할 때는 절대 `assert`를 쓰지 말고, 반드시 확인해야 하는 것에는 진짜 `if ... raise ValueError(...)`를 사용하자.
- `encoding="utf-8"` should be specified explicitly every time a file is opened — relying on the system default can silently produce garbled text (especially with Korean or other non-ASCII content) when a script runs on a different machine.  
파일을 열 때마다 `encoding="utf-8"`을 명시적으로 지정하자 — 시스템 기본값에 의존하면 스크립트가 다른 컴퓨터에서 실행될 때 (특히 한글이나 다른 비-ASCII 콘텐츠에서) 조용히 깨진 텍스트가 나올 수 있다.

---
# 🔗 Related Concepts (관련 개념)

```
[10] OOP Basics
        │
        ▼
[11] Error Handling & File I/O   ← YOU ARE HERE (현재 위치)
        │
        ▼
[12] JS → Python Comparison & Pitfalls
        │
        ▼
[13] Pandas & NumPy for BA
        │
        ▼
[14] Frequently Used Patterns
```

*How is today's topic connected to other concepts? 오늘 배운 주제는 다른 개념과 어떻게 연결되나요?*

A custom exception class is, structurally, just an ordinary class from Section 10 that inherits from `Exception` instead of nothing — everything about `__init__`, inheritance, and `super()` applies directly. The `with` statement itself works through magic methods (`__enter__`/`__exit__`) that any class can implement, the same magic-method mechanism covered in Section 10.

커스텀 예외 클래스는 구조적으로, 아무것도 상속하지 않는 대신 `Exception`을 상속하는 Section 10의 평범한 클래스일 뿐이다 — `__init__`, 상속, `super()`에 관한 모든 것이 그대로 적용된다. `with` 문 자체도 어떤 클래스든 구현할 수 있는 매직 메서드(`__enter__`/`__exit__`)를 통해 동작하며, 이는 Section 10에서 다룬 것과 같은 매직 메서드 메커니즘이다.

pandas' `read_csv()` is, conceptually, exactly what `csv.DictReader` plus a loop does here, scaled up — and pandas raises many of the same exceptions (`FileNotFoundError`, `KeyError` for a missing column) covered in this section, so this `try/except` vocabulary transfers directly once Section 13 arrives.

pandas의 `read_csv()`는 개념적으로, 정확히 여기서 `csv.DictReader`와 반복문이 하는 일을 규모만 키운 것이다 — pandas도 이 섹션에서 다룬 것과 같은 예외(`FileNotFoundError`, 없는 컬럼에 대한 `KeyError`)를 많이 던지므로, Section 13에 도달하면 이 `try/except` 어휘가 그대로 옮겨간다.

---
# 💼 Business Example (비즈니스 예제)
*How would a Business Analyst use this? BA는 이것을 실제로 어떻게 사용할까요?*

**Scenario (시나리오):** A batch of customer records arrived as a CSV export, but a few rows have missing or malformed data. You need to process what you can, skip and log what you cannot, and never let one bad row crash the whole script.

**시나리오:** 고객 레코드 묶음이 CSV 내보내기 형태로 도착했지만, 몇몇 행에 데이터가 없거나 잘못되어 있다. 처리할 수 있는 것은 처리하고, 처리할 수 없는 것은 건너뛰고 기록하며, 잘못된 행 하나가 스크립트 전체를 죽이지 않게 해야 한다.

**To do (할 일 목록):**
- [x] Write records to a CSV file, including some intentionally malformed rows.  
일부러 잘못 만든 행을 포함해 레코드를 CSV 파일에 쓴다.
- [x] Read the CSV back with `csv.DictReader` and process each row inside `try/except`.  
`csv.DictReader`로 CSV를 다시 읽고 각 행을 `try/except` 안에서 처리한다.
- [x] Raise a custom `InvalidRecordError` for rows that fail validation, and catch it separately from parsing errors.  
검증에 실패한 행에는 커스텀 `InvalidRecordError`를 발생시키고, 파싱 에러와 별도로 잡는다.
- [x] Print a final summary of how many rows succeeded vs failed.  
성공한 행과 실패한 행이 각각 몇 개인지 최종 요약을 출력한다.

In [10]:
import csv, os

class InvalidRecordError(Exception):
    pass

demo_path = "customer_batch.csv"

# Write a CSV with a couple of intentionally bad rows mixed in
# 일부러 잘못된 행 몇 개를 섞어 CSV 작성
with open(demo_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age", "email"])
    writer.writeheader()
    writer.writerows([
        {"name": "booki", "age": "28", "email": "booki@example.com"},
        {"name": "Kim",  "age": "abc", "email": "kim@example.com"},          # bad age -- will fail int()
        {"name": "",     "age": "35", "email": "noemail@example.com"},        # missing name -- custom validation failure
        {"name": "Park", "age": "41", "email": "park@example.com"},
    ])

def validate_row(row):
    if not row["name"]:
        raise InvalidRecordError("name is required")
    return {"name": row["name"], "age": int(row["age"]), "email": row["email"]}

processed, failed = [], []

with open(demo_path, "r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        try:
            processed.append(validate_row(row))
        except ValueError as e:
            failed.append((row, f"parsing error: {e}"))
        except InvalidRecordError as e:
            failed.append((row, f"validation error: {e}"))

print("Processed successfully (성공적으로 처리됨):")
for r in processed:
    print(" ", r)

print("Failed rows (실패한 행):")
for row, reason in failed:
    print(" ", row, "->", reason)

print(f"\nSummary: {len(processed)} succeeded, {len(failed)} failed")

os.remove(demo_path)

Processed successfully (성공적으로 처리됨):
  {'name': 'booki', 'age': 28, 'email': 'booki@example.com'}
  {'name': 'Park', 'age': 41, 'email': 'park@example.com'}
Failed rows (실패한 행):
  {'name': 'Kim', 'age': 'abc', 'email': 'kim@example.com'} -> parsing error: invalid literal for int() with base 10: 'abc'
  {'name': '', 'age': '35', 'email': 'noemail@example.com'} -> validation error: name is required

Summary: 2 succeeded, 2 failed


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences. 오늘 배운 개념을 3~5문장으로 정리해보세요.*

Python organizes error handling around `try/except` blocks that catch specific exception classes — `ValueError`, `KeyError`, `FileNotFoundError`, and many more — rather than one generic error type, with `else` running only when no exception occurred and `finally` always running for cleanup. `raise` triggers an exception on purpose, a bare `raise` inside an `except` block re-raises the current one, and a custom exception is just a class that inherits from `Exception`, useful once a built-in exception is not specific enough to describe what went wrong. `assert` is a lightweight sanity check for catching bugs during development, not a substitute for validating real user input. File I/O centers on `with open(path, mode, encoding="utf-8") as f:`, which guarantees the file closes automatically even if an error happens partway through — reading options range from `.read()` (the whole file as one string) to iterating line by line (memory-efficient for large files), and the three most common modes are `"r"` (read), `"w"` (overwrite), and `"a"` (append). `csv.DictReader`/`csv.DictWriter` read and write CSV rows as dictionaries keyed by header, but every value comes back as a `str` and must be converted manually. Both catching narrow exception types and using `with` consistently are two of the highest-leverage habits for writing scripts that survive messy real-world data.

Python은 하나의 범용 에러 타입이 아니라 `ValueError`, `KeyError`, `FileNotFoundError`를 비롯한 특정 예외 클래스를 잡는 `try/except` 블록을 중심으로 에러 처리를 조직한다. `else`는 예외가 발생하지 않았을 때만 실행되고, `finally`는 항상 정리를 위해 실행된다. `raise`는 의도적으로 예외를 발생시키고, `except` 블록 안의 단독 `raise`는 현재 예외를 재발생시키며, 커스텀 예외는 내장 예외가 무엇이 잘못되었는지 설명하기에 충분히 구체적이지 않을 때 유용한, `Exception`을 상속하는 클래스일 뿐이다. `assert`는 실제 사용자 입력을 검증하는 대체물이 아니라 개발 중 버그를 잡기 위한 가벼운 확인이다. 파일 I/O는 `with open(path, mode, encoding="utf-8") as f:`를 중심으로 돌아가며, 이는 도중에 에러가 발생해도 파일이 자동으로 닫히는 것을 보장한다 — 읽기 옵션은 `.read()`(전체를 하나의 문자열로)부터 한 줄씩 순회하기(큰 파일에 메모리 효율적)까지 다양하며, 가장 흔한 세 가지 모드는 `"r"`(읽기), `"w"`(덮어쓰기), `"a"`(추가)다. `csv.DictReader`/`csv.DictWriter`는 헤더를 키로 하는 딕셔너리로 CSV 행을 읽고 쓰지만, 모든 값은 `str`로 돌아오므로 직접 변환해야 한다. 좁은 예외 타입을 잡는 것과 `with`를 일관되게 쓰는 것 둘 다, 지저분한 실제 데이터에서도 살아남는 스크립트를 작성하는 데 가장 효율 높은 습관이다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. 오늘의 주제를 한 문장으로.

> Python catches specific exception types with `try/except/else/finally` (raising custom ones when built-ins are not specific enough) and reads/writes files safely through `with open(...) as f:`, which guarantees the file closes automatically — with CSV values always arriving as strings that need manual conversion.

> Python은 `try/except/else/finally`로 특정 예외 타입을 잡고(내장 예외가 충분히 구체적이지 않으면 커스텀 예외를 발생시키며), `with open(...) as f:`를 통해 파일이 자동으로 닫히는 것을 보장하며 안전하게 파일을 읽고 쓴다 — CSV 값은 항상 문자열로 도착하므로 직접 변환이 필요하다.

---
# ❓ Review Questions (복습 문제)

**Q1.** When does the `else` block in a `try/except/else` statement run, and how is that different from `finally`?  
`try/except/else` 문의 `else` 블록은 언제 실행되나요? `finally`와는 어떻게 다른가요?

**Q2.** Why is `except Exception:` considered risky if used everywhere, instead of catching a more specific exception type?  
더 구체적인 예외 타입을 잡는 대신 어디서나 `except Exception:`을 쓰는 것이 왜 위험하다고 여겨지나요?

**Q3.** What does a bare `raise` (with no exception specified) do inside an `except` block?  
`except` 블록 안에서 예외를 지정하지 않은 단독 `raise`는 무엇을 하나요?

**Q4.** Why does `with open(path) as f:` help avoid bugs compared to calling `open()` and `f.close()` manually?  
`open()`과 `f.close()`를 수동으로 호출하는 것과 비교해 `with open(path) as f:`는 왜 버그를 피하는 데 도움이 되나요?

**Q5.** After reading a column of ages from a CSV with `csv.DictReader`, what type is each age value, and what must you do before doing math with it?  
`csv.DictReader`로 CSV에서 나이 컬럼을 읽은 뒤, 각 나이 값은 어떤 타입인가요? 그것으로 계산을 하기 전에 무엇을 해야 하나요?

---
*📅 Try answering these again in a few days. 며칠 후 다시 풀어보세요.*